<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/lstm/notebooks/04_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - LSTM Model

## 1. Connecting to the repository and Google Drive

In [ ]:
!pip install "numpy<2" "pandas>=2.0.0" -q

In [1]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')


Already up to date.
/content/recommendation-system
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check out the right branch

In [2]:
!git fetch origin lstm
!git checkout -B lstm origin/lstm


From https://github.com/mkobycheva/recommendation-system
 * branch            lstm       -> FETCH_HEAD
Branch 'lstm' set up to track remote branch 'lstm' from 'origin'.
Reset branch 'lstm'
Your branch is up to date with 'origin/lstm'.


## 2. Prepare data

In [4]:
import os
import numpy as np
import pandas as pd

from configs.model_configs import LSTM_CONFIG
from src.data.build_matrix import add_domain_item_ids
from src.evaluation.cross_domain_eval import evaluate_multi_domain

DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')


In [5]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()


,user_id,item_id,domain
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0061713244,books
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0871139634,books
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0385521685,books
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0848732634,books
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,books::0151014981,books


In [6]:
unique_train_items = train_df['item_id'].unique()

item_to_idx = {item: idx + 1 for idx, item in enumerate(unique_train_items)}

UNK_IDX = len(item_to_idx) + 1

def encode_item_ids(df, mapping, unk_value):
    return df['item_id'].map(mapping).fillna(unk_value).astype(int)

train_df['item_idx'] = encode_item_ids(train_df, item_to_idx, UNK_IDX)
valid_df['item_idx'] = encode_item_ids(valid_df, item_to_idx, UNK_IDX)
test_df['item_idx'] = encode_item_ids(test_df, item_to_idx, UNK_IDX)

NUM_ITEMS = UNK_IDX + 1
print(f"Number of unique items for LSTM (including padding & UNK): {NUM_ITEMS}")

Number of unique items for LSTM (including padding & UNK): 587066


In [7]:
def pad_sequence(seq, max_len):
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [0] * (max_len - len(seq)) + seq


def generate_train_sequences(df, max_len):
    X, y = [], []
    df_sorted = df.sort_values(by=['user_id', 'timestamp'])

    for _, group in df_sorted.groupby('user_id'):
        item_list = group['item_idx'].tolist()
        if len(item_list) < 2:
            continue

        for pos in range(1, len(item_list)):
            X.append(pad_sequence(item_list[:pos], max_len))
            y.append(item_list[pos])

    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)


def generate_holdout_sequences(history_df, holdout_df, max_len):
    X, y = [], []
    combined = pd.concat([
        history_df.assign(is_holdout=0),
        holdout_df.assign(is_holdout=1),
    ], ignore_index=True).sort_values(by=['user_id', 'timestamp'])

    for _, group in combined.groupby('user_id'):
        item_list = group['item_idx'].tolist()
        holdout_flags = group['is_holdout'].tolist()
        for pos in range(1, len(item_list)):
            if holdout_flags[pos] == 1:
                X.append(pad_sequence(item_list[:pos], max_len))
                y.append(item_list[pos])

    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)


MAX_LEN = LSTM_CONFIG['max_len']

print('Make orders for Train...')
X_train, y_train = generate_train_sequences(train_df, max_len=MAX_LEN)

print('Make orders for Validation...')
X_val, y_val = generate_holdout_sequences(train_df, valid_df, max_len=MAX_LEN)

print('Make orders for Test...')
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
X_test, y_test = generate_holdout_sequences(train_valid_df, test_df, max_len=MAX_LEN)

print(f'Train: X = {X_train.shape}, y = {y_train.shape}')
print(f'Valid: X = {X_val.shape}, y = {y_val.shape}')
print(f'Test:  X = {X_test.shape}, y = {y_test.shape}')


Make orders for Train...
Make orders for Validation...
Make orders for Test...
Train: X = (3623625, 10), y = (3623625,)
Valid: X = (254376, 10), y = (254376,)
Test:  X = (254376, 10), y = (254376,)


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class RecSysDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loaders(X_train, y_train, X_val, y_val):
    train_dataset = RecSysDataset(X_train, y_train)
    val_dataset = RecSysDataset(X_val, y_val)
    batch_size = LSTM_CONFIG['batch_size']
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
    )


train_loader, val_loader = make_loaders(X_train, y_train, X_val, y_val)

for sample_x, sample_y in train_loader:
    print(f'Batch inputs (X): {sample_x.shape}')
    print(f'Batch correct outputs (y): {sample_y.shape}')
    break


Batch inputs (X): torch.Size([512, 10])
Batch correct outputs (y): torch.Size([512])


## 3. Main part

In [9]:
import torch.nn as nn

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip install lightning -q

In [12]:
def calculate_map_at_5(logits, targets):
    _, top5_indices = torch.topk(logits, k=5, dim=-1)

    targets_expanded = targets.unsqueeze(-1).expand_as(top5_indices)
    hits = (top5_indices == targets_expanded)

    ranks = torch.arange(1, 6, device=logits.device).float()

    ap_scores = (hits.float() / ranks).sum(dim=-1)

    return ap_scores.mean().item()

In [13]:
import lightning as L

class LightningRecSysLSTM(L.LightningModule):
    def __init__(
        self,
        num_items,
        embedding_dim=None,
        hidden_dim=None,
        lr=None,
        weight_decay=None,
        dropout=None,
    ):
        super().__init__()
        self.save_hyperparameters()

        embedding_dim = embedding_dim or LSTM_CONFIG['embedding_dim']
        hidden_dim = hidden_dim or LSTM_CONFIG['hidden_dim']
        dropout = LSTM_CONFIG['dropout'] if dropout is None else dropout
        self.lr = LSTM_CONFIG['lr'] if lr is None else lr
        self.weight_decay = LSTM_CONFIG['weight_decay'] if weight_decay is None else weight_decay

        self.item_embeddings = nn.Embedding(num_items, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_items)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)

    def forward(self, x):
        embedded = self.item_embeddings(x)
        out, _ = self.lstm(embedded)
        last_step = self.dropout(out[:, -1, :])
        return self.fc(last_step)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)
        val_map = calculate_map_at_5(logits, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_map_at_5', val_map, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )


In [15]:
import os
import time
from lightning.pytorch.loggers import WandbLogger
import wandb

wandb.finish()

REAL_WANDB_KEY = ""

netrc_path = os.path.expanduser("~/.netrc")
with open(netrc_path, "w") as f:
    f.write(f"machine api.wandb.ai\n  login user\n  password {REAL_WANDB_KEY}\n")

os.environ["WANDB_API_KEY"] = REAL_WANDB_KEY
os.environ["WANDB_RESUME"] = "never"

unique_run_id = f"lstm-10epochs-final-{int(time.time())}"

wandb_logger = WandbLogger(
    project='recsys-amazon-lstm',
    log_model='all',
    id=unique_run_id
)

drive_model_dir = '/content/drive/MyDrive/recsys-data/saved_models'

trainer = L.Trainer(
    max_epochs=10,
    accelerator='auto',
    logger=wandb_logger,
    enable_checkpointing=True,
    default_root_dir=drive_model_dir,
)

lightning_model = LightningRecSysLSTM(num_items=NUM_ITEMS).to(device)
wandb_logger.watch(lightning_model, log='all', log_freq=100)

trainer.fit(lightning_model, train_loader, val_loader)

best_checkpoint_path = trainer.checkpoint_callback.best_model_path
print(f"🔥 Успішно! Найкраща модель збережена тут: {best_checkpoint_path}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rsai2026_2 (rsai2026_2-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relo

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ item_embeddings │ Embedding        │ 37.6 M │ train │     0 │
│ 1 │ lstm            │ LSTM             │ 99.3 K │ train │     0 │
│ 2 │ dropout         │ Dropout          │      0 │ train │     0 │
│ 3 │ fc              │ Linear           │ 75.7 M │ train │     0 │
│ 4 │ criterion       │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 113 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 113 M                                                                                                
Total estimated model params size (MB): 453.612                                                                    
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


🔥 Успішно! Найкраща модель збережена тут: ./recsys-amazon-lstm/lstm-10epochs-final-1784015908/checkpoints/epoch=9-step=70780.ckpt


In [25]:
CHECKPOINT_PATH = "./recsys-amazon-lstm/lstm-10epochs-final-1784015908/checkpoints/epoch=9-step=70780.ckpt"

lightning_model = LightningRecSysLSTM.load_from_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    num_items=NUM_ITEMS
)

lightning_model.to(device)
lightning_model.eval()

print("✅ Навчену модель успішно завантажено!")

✅ Навчену модель успішно завантажено!


In [20]:
#!cp ./recsys-amazon-lstm/lstm-10epochs-final-1784015908/checkpoints/epoch=9-step=70780.ckpt /content/drive/MyDrive/recsys-data/saved_models/

## 4. Evaluate model

In [21]:
idx_to_item = {idx: item for item, idx in item_to_idx.items()}


def domain_indices_from_mapping(item_to_idx, domains=('books', 'movies')):
    return {
        domain: np.array([
            idx for item, idx in item_to_idx.items()
            if item.startswith(f'{domain}::')
        ], dtype=np.int64)
        for domain in domains
    }


def train_seen_idx_by_user(train_interactions_df):
    return (
        train_interactions_df.groupby('user_id')['item_idx']
        .agg(lambda values: set(int(v) for v in values if int(v) != UNK_IDX))
        .to_dict()
    )


def user_history_by_train(train_interactions_df, max_len):
    histories = {}
    train_sorted = train_interactions_df.sort_values(by=['user_id', 'timestamp'])
    for user_id, group in train_sorted.groupby('user_id'):
        histories[user_id] = pad_sequence(group['item_idx'].tolist(), max_len)
    return histories


def make_lstm_recommender(trained_lightning_model, train_interactions_df):
    trained_lightning_model.eval()
    trained_lightning_model.to(device)
    domain_item_indices = domain_indices_from_mapping(item_to_idx)
    seen_by_user = train_seen_idx_by_user(train_interactions_df)
    histories = user_history_by_train(train_interactions_df, LSTM_CONFIG['max_len'])

    def recommend_for_users(eval_users, target_domain, k=10):
        recommendations = {}
        eval_users = list(eval_users)
        candidate_indices = domain_item_indices.get(target_domain, np.array([], dtype=np.int64))
        if len(candidate_indices) == 0:
            return {user_id: [] for user_id in eval_users}

        base_mask = torch.full((NUM_ITEMS,), -torch.inf, dtype=torch.float, device=device)
        base_mask[torch.tensor(candidate_indices, dtype=torch.long, device=device)] = 0.0
        base_mask[0] = -torch.inf
        base_mask[UNK_IDX] = -torch.inf

        batch_size = LSTM_CONFIG['batch_size']
        for start in range(0, len(eval_users), batch_size):
            batch_users = eval_users[start:start + batch_size]
            X_batch_list = [histories.get(user_id, [0] * LSTM_CONFIG['max_len']) for user_id in batch_users]
            X_batch = torch.tensor(X_batch_list, dtype=torch.long, device=device)

            with torch.no_grad():
                logits = trained_lightning_model(X_batch)
                masked_logits = logits + base_mask.unsqueeze(0)
                for row_idx, user_id in enumerate(batch_users):
                    seen = seen_by_user.get(user_id, set())
                    if seen:
                        seen_idx = torch.tensor(list(seen), dtype=torch.long, device=device)
                        masked_logits[row_idx, seen_idx] = -torch.inf
                batch_top_indices = []
                for row_idx in range(masked_logits.shape[0]):
                    finite_count = int(torch.isfinite(masked_logits[row_idx]).sum().item())
                    if finite_count == 0:
                        batch_top_indices.append([])
                        continue
                    top_n = min(k, finite_count)
                    top_indices = torch.topk(masked_logits[row_idx], k=top_n).indices.cpu().numpy().tolist()
                    batch_top_indices.append(top_indices)

            for row_idx, user_id in enumerate(batch_users):
                recommendations[user_id] = [
                    idx_to_item[idx]
                    for idx in batch_top_indices[row_idx]
                    if idx in idx_to_item
                ]

        return recommendations

    return recommend_for_users


recommend_for_users = make_lstm_recommender(lightning_model, train_df)


In [22]:
def lstm_result_row(model_name, train_interactions_df, valid_results, test_results):
    row = {
        'model': model_name,
        'embedding_dim': LSTM_CONFIG['embedding_dim'],
        'hidden_dim': LSTM_CONFIG['hidden_dim'],
        'max_len': LSTM_CONFIG['max_len'],
        'batch_size': LSTM_CONFIG['batch_size'],
        'lr': LSTM_CONFIG['lr'],
        'weight_decay': LSTM_CONFIG['weight_decay'],
        'dropout': LSTM_CONFIG['dropout'],
        'epochs': LSTM_CONFIG['epochs'],
        'k': LSTM_CONFIG['k'],
        'n_users': train_interactions_df['user_id'].nunique(),
        'n_items': train_interactions_df['item_id'].nunique(),
        'n_train_interactions': len(train_interactions_df),
    }
    for split_name, results in [('valid', valid_results), ('test', test_results)]:
        for metric in ['ndcg@10', 'recall@10', 'map@10']:
            values = []
            for domain in ['books', 'movies']:
                value = results.get(domain, {}).get(metric, np.nan)
                row[f'{domain}_{split_name}_{metric}'] = value
                if not pd.isna(value):
                    values.append(value)
            row[f'{split_name}_macro_{metric}'] = round(float(np.mean(values)), 4) if values else np.nan
    return row


In [23]:
K = LSTM_CONFIG['k']

valid_results, test_results = evaluate_multi_domain(
    split_dfs={'valid': valid_df, 'test': test_df},
    recommend_func=recommend_for_users,
    k=K,
)

joint_result_row = lstm_result_row(
    'lstm_books_movies_joint',
    train_df,
    valid_results,
    test_results,
)

for split_name, results in [('valid', valid_results), ('test', test_results)]:
    print(f'\n{split_name.upper()}')
    for domain, metrics in results.items():
        print(f"{domain}: NDCG@10={metrics['ndcg@10']}, Recall@10={metrics['recall@10']}, MAP@10={metrics['map@10']}")

joint_result_row


[Evaluation] Scoring BOOKS: 127,188 unique users across splits
  -> VALID Results:
     MAP@10: 0.0071 | NDCG@10: 0.0096 | Recall@10: 0.0177
  -> TEST Results:
     MAP@10: 0.0046 | NDCG@10: 0.0062 | Recall@10: 0.0116
[Evaluation] Scoring MOVIES: 127,188 unique users across splits
  -> VALID Results:
     MAP@10: 0.0162 | NDCG@10: 0.0212 | Recall@10: 0.0376
  -> TEST Results:
     MAP@10: 0.0098 | NDCG@10: 0.0131 | Recall@10: 0.0239

VALID
books: NDCG@10=0.0096, Recall@10=0.0177, MAP@10=0.0071
movies: NDCG@10=0.0212, Recall@10=0.0376, MAP@10=0.0162

TEST
books: NDCG@10=0.0062, Recall@10=0.0116, MAP@10=0.0046
movies: NDCG@10=0.0131, Recall@10=0.0239, MAP@10=0.0098


{'model': 'lstm_books_movies_joint',
 'embedding_dim': 64,
 'hidden_dim': 128,
 'max_len': 10,
 'batch_size': 512,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'dropout': 0.2,
 'epochs': 5,
 'k': 10,
 'n_users': 127188,
 'n_items': 587064,
 'n_train_interactions': 3750813,
 'books_valid_ndcg@10': 0.0096,
 'movies_valid_ndcg@10': 0.0212,
 'valid_macro_ndcg@10': 0.0154,
 'books_valid_recall@10': 0.0177,
 'movies_valid_recall@10': 0.0376,
 'valid_macro_recall@10': 0.0277,
 'books_valid_map@10': 0.0071,
 'movies_valid_map@10': 0.0162,
 'valid_macro_map@10': 0.0117,
 'books_test_ndcg@10': 0.0062,
 'movies_test_ndcg@10': 0.0131,
 'test_macro_ndcg@10': 0.0097,
 'books_test_recall@10': 0.0116,
 'movies_test_recall@10': 0.0239,
 'test_macro_recall@10': 0.0178,
 'books_test_map@10': 0.0046,
 'movies_test_map@10': 0.0098,
 'test_macro_map@10': 0.0072}

## 5. Single-Domain model

In [ ]:
import os
import pandas as pd
import lightning as L

EPOCHS = 10

def fit_lstm_model_for_domain(domain_train_df, domain_valid_df):
    X_domain_train, y_domain_train = generate_train_sequences(domain_train_df, max_len=LSTM_CONFIG['max_len'])
    X_domain_val, y_domain_val = generate_holdout_sequences(domain_train_df, domain_valid_df, max_len=LSTM_CONFIG['max_len'])

    domain_train_loader, domain_val_loader = make_loaders(
        X_domain_train,
        y_domain_train,
        X_domain_val,
        y_domain_val,
    )

    domain_model = LightningRecSysLSTM(num_items=NUM_ITEMS).to(device)

    domain_trainer = L.Trainer(
        max_epochs=EPOCHS,
        accelerator='auto',
        logger=False,
        enable_checkpointing=False,
        default_root_dir=drive_model_dir,
    )

    domain_trainer.fit(domain_model, domain_train_loader, domain_val_loader)
    return domain_model


books_train_indexed = train_df[train_df['domain'] == 'books']
movies_train_indexed = train_df[train_df['domain'] == 'movies']


books_only_model = fit_lstm_model_for_domain(
    books_train_indexed,
    valid_df[valid_df['domain'] == 'books'],
)

books_only_recommend_for_users = make_lstm_recommender(books_only_model, books_train_indexed)
books_only_valid_results, books_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'books'],
        'test': test_df[test_df['domain'] == 'books'],
    },
    recommend_func=books_only_recommend_for_users,
    k=K,
)

books_only_result_row = lstm_result_row(
    'lstm_books_only',
    books_train_indexed,
    books_only_valid_results,
    books_only_test_results,
)


movies_only_model = fit_lstm_model_for_domain(
    movies_train_indexed,
    valid_df[valid_df['domain'] == 'movies'],
)


movies_only_recommend_for_users = make_lstm_recommender(movies_only_model, movies_train_indexed)
movies_only_valid_results, movies_only_test_results = evaluate_multi_domain(
    split_dfs={
        'valid': valid_df[valid_df['domain'] == 'movies'],
        'test': test_df[test_df['domain'] == 'movies'],
    },
    recommend_func=movies_only_recommend_for_users,
    k=K,
)

movies_only_result_row = lstm_result_row(
    'lstm_movies_only',
    movies_train_indexed,
    movies_only_valid_results,
    movies_only_test_results,
)


lstm_ablation_results = pd.DataFrame([
    books_only_result_row,
    movies_only_result_row,
    joint_result_row,
])

os.makedirs('results', exist_ok=True)
lstm_ablation_results.to_csv('results/lstm_ablation_metrics.csv', index=False)

print("\nРезультати збережено в 'results/lstm_ablation_metrics.csv'")
lstm_ablation_results

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ item_embeddings │ Embedding        │ 37.6 M │ train │     0 │
│ 1 │ lstm            │ LSTM             │ 99.3 K │ train │     0 │
│ 2 │ dropout         │ Dropout          │      0 │ train │     0 │
│ 3 │ fc              │ Linear           │ 75.7 M │ train │     0 │
│ 4 │ criterion       │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 113 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 113 M                                                                                                
Total estimated model params size (MB): 453.612                                                                    
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.